In [1]:
from datetime import timedelta

# Frequencies of nodes per hour at ATB and HSK
freq1 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 2, 0, 0]  # ATB
freq2 = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 4, 5, 6, 4, 0, 0]  # HSK

# freq1 = [0, 0, 0, 0, 3, 3, 4, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]
# freq2 = [0, 0, 0, 0, 4, 6 ,5, 8, 10, 10, 12, 16, 9, 15, 15, 16, 14, 12, 9, 6, 6, 4, 0, 0]

# Generate nodes with ids, times, and locations
nodes = []
node_id = 1

# Add start node at 00:00 (location 'X')
nodes.append({'id': node_id, 'time': 0, 'loc': 'X'})
start_node_id = node_id
node_id += 1

for hour in range(24):
    f1 = freq1[hour]
    f2 = freq2[hour]

    # Generate ATB nodes for this hour
    for i in range(f1):
        time_val = hour * 60 + (i * 60 // max(f1, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'ATB'})
        node_id += 1

    # Generate HSK nodes for this hour
    for i in range(f2):
        time_val = hour * 60 + (i * 60 // max(f2, 1))
        nodes.append({'id': node_id, 'time': time_val, 'loc': 'HSK'})
        node_id += 1

# Add end node at 24:00 (location 'X')
nodes.append({'id': node_id, 'time': 1440, 'loc': 'X'})
node_id += 1

# Sort nodes by time ascending
nodes.sort(key=lambda x: x['time'])

# Constants for arc generation
MinUtilTime = 400

# Travel times and capacities for arcs (in minutes)
arc_data = {
    ("X", "X"): (0, 150),
    ("X", "HSK"): (10, 150),
    ("X", "ATB"): (40, 150),
    ("HSK", "HSK"): (200, 150),
    ("ATB", "ATB"): (200, 150),
    ("HSK", "ATB"): (110, 1),
    ("ATB", "HSK"): (110, 1),
    ("HSK", "X"): (10, 150),
    ("ATB", "X"): (40, 150)
}

transitions = {
    "ATB": "HSK",
    "HSK": "ATB"
}

travel_time = {
    ("ATB", "HSK"): arc_data[("ATB", "HSK")][0],
    ("HSK", "ATB"): arc_data[("HSK", "ATB")][0]
}

synthetic_id = len(nodes) + 1
E = []

# Arcs from 'X' to all ATB/HSK nodes (if reachable)
for node in nodes:
    loc = node['loc']
    dst_time = node['time']

    if loc not in ['ATB', 'HSK']:
        continue

    cost = arc_data[('X', loc)][0]
    cap = arc_data[('X', loc)][1]
    T=dst_time-cost

    if dst_time >= cost:
        E.append({
            'src': {'id': start_node_id, 'time': T, 'loc': 'X'},
            'dst': {'id': node['id'], 'time': dst_time, 'loc': loc},
            'cost': cost,
            'cap': cap
        })

# Chained arcs
for node in nodes:
    loc = node['loc']
    time = node['time']
    src_id = node['id']

    if loc not in transitions:
        continue

    total_time = 0
    curr_loc = loc
    curr_time = time
    curr_src_id = src_id

    while total_time < MinUtilTime:
        next_loc = transitions[curr_loc]
        cost = travel_time[(curr_loc, next_loc)]
        next_time = curr_time + cost

        if next_time > 1440:
            break

        dst = {'id': synthetic_id, 'time': next_time, 'loc': next_loc}
        synthetic_id += 1

        E.append({
            'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
            'dst': dst,
            'cost': cost,
            'cap': 1
        })

        total_time += cost
        curr_loc = next_loc
        curr_time = next_time
        curr_src_id = dst['id']

    cost_to_X, cap_to_X = arc_data[(curr_loc, 'X')]
    E.append({
        'src': {'id': curr_src_id, 'time': curr_time, 'loc': curr_loc},
        'dst': {'id': synthetic_id, 'time': curr_time + cost_to_X, 'loc': 'X'},
        'cost': cost_to_X,
        'cap': cap_to_X
    })
    synthetic_id += 1

for node in nodes:
    node['hhmm'] = str(timedelta(minutes=node['time']))[:-3]

# print(E)
count=0
# Output preview
for node in nodes:
    # if(node['loc']=='HSK'):
        print(node)
        count+=1
print(count)

# Print count and sample arcs
print(f"Total generated arcs: {len(E)}")
for arc in E[:]:
    print(f"From node {arc['src']['id']} ({arc['src']['loc']} @ {arc['src']['time']}) "
          f"to node {arc['dst']['id']} ({arc['dst']['loc']} @ {arc['dst']['time']}) "
          f"cost: {arc['cost']}")

{'id': 1, 'time': 0, 'loc': 'X', 'hhmm': '0:00'}
{'id': 2, 'time': 1020, 'loc': 'ATB', 'hhmm': '17:00'}
{'id': 5, 'time': 1020, 'loc': 'HSK', 'hhmm': '17:00'}
{'id': 3, 'time': 1040, 'loc': 'ATB', 'hhmm': '17:20'}
{'id': 6, 'time': 1040, 'loc': 'HSK', 'hhmm': '17:20'}
{'id': 4, 'time': 1060, 'loc': 'ATB', 'hhmm': '17:40'}
{'id': 7, 'time': 1060, 'loc': 'HSK', 'hhmm': '17:40'}
{'id': 8, 'time': 1080, 'loc': 'ATB', 'hhmm': '18:00'}
{'id': 12, 'time': 1080, 'loc': 'HSK', 'hhmm': '18:00'}
{'id': 9, 'time': 1095, 'loc': 'ATB', 'hhmm': '18:15'}
{'id': 13, 'time': 1095, 'loc': 'HSK', 'hhmm': '18:15'}
{'id': 10, 'time': 1110, 'loc': 'ATB', 'hhmm': '18:30'}
{'id': 14, 'time': 1110, 'loc': 'HSK', 'hhmm': '18:30'}
{'id': 11, 'time': 1125, 'loc': 'ATB', 'hhmm': '18:45'}
{'id': 15, 'time': 1125, 'loc': 'HSK', 'hhmm': '18:45'}
{'id': 16, 'time': 1140, 'loc': 'ATB', 'hhmm': '19:00'}
{'id': 21, 'time': 1140, 'loc': 'HSK', 'hhmm': '19:00'}
{'id': 17, 'time': 1152, 'loc': 'ATB', 'hhmm': '19:12'}
{'id': 